# Mamba3Yolo — Plug-and-Play v3 (NaN-safe)

**Repo**: https://github.com/ShMazumder/Mamba3Yolo

1. Run All  
2. Restart Kernel when told  
3. Run All again

### v3 fixes
- NaN-safe: AMP off, LR=3e-4, grad_clip=1.0, stable loss, NaN skip
- Permanent pure-PyTorch fallback after first kernel failure
- SISO preferred
- Auto-detects brain-tumor dataset

## 1. Install (then Restart Kernel)

In [ ]:
import subprocess, sys
def pip(*a):
    return subprocess.run([sys.executable,"-m","pip","install","-q","--no-cache-dir"]+list(a), capture_output=True, text=True)
print("1/3 Base packages...")
pip("einops","thop","seaborn","timm","opencv-python-headless","torchmetrics","pycocotools")
print("   done")
print("2/3 Mamba-3 kernels...")
r = pip("causal-conv1d>=1.4.0","mamba-ssm","--no-build-isolation")
print("   ✅" if r.returncode==0 else "   ⚠️ fallback will be used")
print("3/3 Ready.\n"+"="*60+"\n>>> RESTART KERNEL NOW\n>>> Then Run All again\n"+"="*60)

## 2. Setup + Auto-detect

In [ ]:
import os, sys, gc, time, json, random, warnings, math
from pathlib import Path
from datetime import datetime
warnings.filterwarnings("ignore")
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from IPython.display import display

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE=="cuda": print(f"GPU: {torch.cuda.get_device_name(0)}")

WORK=Path("/kaggle/working"); PROJ=WORK/"Mamba3Yolo"
if not PROJ.exists():
    !git clone --depth 1 https://github.com/ShMazumder/Mamba3Yolo.git {PROJ}
else:
    print("Repo present")
sys.path.insert(0,str(PROJ)); os.chdir(PROJ)
print("Project:", PROJ)
print("Block exists:", (PROJ/"src"/"blocks"/"mamba3_odss.py").exists())

def find_images_dir(root, max_depth=4):
    root=Path(root)
    if not root.exists(): return None
    for cand in [root/"images"/"train", root/"train"/"images", root/"train",
                 root/"images"/"train2017", root/"train2017", root/"images",
                 root/"images"/"val", root/"valid"/"images", root/"val", root]:
        if cand.exists() and (any(cand.glob("*.jpg")) or any(cand.glob("*.png"))):
            return cand
    for dirpath, dirnames, filenames in os.walk(root):
        if len(Path(dirpath).relative_to(root).parts)>max_depth: dirnames.clear(); continue
        if any(f.lower().endswith((".jpg",".jpeg",".png")) for f in filenames):
            return Path(dirpath)
    return None

def find_labels_dir(img_dir, root):
    if img_dir is None: return None
    for c in [img_dir.parent/"labels", img_dir.parent.parent/"labels", root/"labels"/"train", root/"labels", root/"train"/"labels", img_dir.with_name("labels")]:
        if c and c.exists(): return c
    return img_dir

CANDIDATES = {
    "coco": ["/kaggle/input/coco-dataset-for-yolo","/kaggle/input/coco-2017-yolo","/kaggle/input/coco-yolo","/kaggle/input/coco2017"],
    "visdrone": ["/kaggle/input/visdrone-dataset","/kaggle/input/visdrone-dataset-yolo-format","/kaggle/input/visdrone-yolo","/kaggle/input/visdrone2019"],
    "br35h": ["/kaggle/input/datasets/organizations/ultralytics/brain-tumor","/kaggle/input/brain-tumor-yolo","/kaggle/input/brain-tumour-br35h","/kaggle/input/br35h","/kaggle/input/brain-tumor"],
    "polyp": ["/kaggle/input/yolo-kvasir","/kaggle/input/kvasirseg","/kaggle/input/kvasir-seg"],
    "bccd": ["/kaggle/input/bccd-yolo","/kaggle/input/blood-cell-detection-yolo","/kaggle/input/bccd"],
}
def find_dataset(name):
    for p in CANDIDATES.get(name,[]):
        p=Path(p)
        if p.exists():
            img_dir=find_images_dir(p)
            if img_dir is not None: return p, img_dir
    return None, None

FOUND={k: {"root":r,"img_dir":i} for k,(r,i) in ((k,find_dataset(k)) for k in CANDIDATES)}
print("\n📦 Auto-detected datasets:")
for k,v in FOUND.items():
    print(f"  {k:10s}: {'✅ '+str(v['root']) if v['root'] else '❌ not added'}")
    if v['img_dir']: print(f"             images → {v['img_dir']}")

if FOUND["coco"]["root"]:
    PRIMARY,DATA_ROOT,NC,IMG_DIR="coco",FOUND["coco"]["root"],80,FOUND["coco"]["img_dir"]
elif FOUND["visdrone"]["root"]:
    PRIMARY,DATA_ROOT,NC,IMG_DIR="visdrone",FOUND["visdrone"]["root"],10,FOUND["visdrone"]["img_dir"]
elif FOUND["br35h"]["root"] or FOUND["polyp"]["root"] or FOUND["bccd"]["root"]:
    PRIMARY="medical"; NC=9
    for k in ["br35h","polyp","bccd"]:
        if FOUND[k]["root"]:
            DATA_ROOT,IMG_DIR=FOUND[k]["root"],FOUND[k]["img_dir"]; break
else:
    PRIMARY,DATA_ROOT,IMG_DIR,NC="synthetic",Path("dummy"),None,80

print(f"\n🎯 Primary: {PRIMARY.upper()} | nc={NC}")
print(f"   data_root = {DATA_ROOT}")
if IMG_DIR: print(f"   img_dir   = {IMG_DIR}")

## 3. Verify Model

In [ ]:
import importlib
import src.blocks.mamba3_odss as mamba_mod
importlib.reload(mamba_mod)
from src.blocks.mamba3_odss import HAS_MAMBA3
from src.models.mamba3yolo import build_mamba3yolo

print("="*55)
print(f"Official Mamba-3 kernels : {HAS_MAMBA3}")
print("="*55)

model = build_mamba3yolo("T", nc=NC, is_mimo=False, d_state=64).to(DEVICE)
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

x = torch.randn(2, 3, 320, 320, device=DEVICE)
with torch.no_grad():
    outs = model(x)
print("Output shapes:", [tuple(o.shape) for o in outs])
del x, outs; gc.collect()
if DEVICE=="cuda": torch.cuda.empty_cache()
print("✅ Model OK")

## 4. Config (NaN-safe)

In [ ]:
CFG = {
    "scale": "T", "nc": NC, "imgsz": 640,
    "epochs": 30 if PRIMARY=="synthetic" else 50,
    "batch_size": 4 if DEVICE=="cuda" else 2,
    "lr": 3e-4, "weight_decay": 1e-2,
    "is_mimo": False, "d_state": 64,
    "amp": False,
    "num_workers": 2, "seed": SEED,
    "project": str(WORK/"runs"/"mamba3yolo"),
    "exp_name": f"{PRIMARY}_{datetime.now().strftime('%m%d_%H%M')}",
    "data_root": str(DATA_ROOT),
    "img_dir": str(IMG_DIR) if IMG_DIR else None,
    "max_train_samples": 2500 if PRIMARY!="synthetic" else None,
    "max_val_samples": 300,
    "primary": PRIMARY, "grad_clip": 1.0,
}
print(json.dumps({k:CFG[k] for k in ["primary","nc","epochs","batch_size","lr","amp","is_mimo","exp_name"]}, indent=2))
os.makedirs(CFG["project"], exist_ok=True)

## 5. Dataset

In [ ]:
class YoloFolderDataset(Dataset):
    def __init__(self, root, img_size=640, max_samples=None, is_train=True, split="train", img_dir_hint=None):
        self.root=Path(root); self.img_size=img_size; self.is_train=is_train
        if img_dir_hint and Path(img_dir_hint).exists():
            self.img_dir=Path(img_dir_hint)
            if not is_train:
                for alt in [self.img_dir.parent/"valid", self.img_dir.parent/"val",
                            self.img_dir.parent/"images"/"val", self.img_dir.parent/"images"/"valid"]:
                    if alt.exists() and (any(alt.glob("*.jpg")) or any(alt.glob("*.png"))):
                        self.img_dir=alt; break
        else:
            self.img_dir=find_images_dir(self.root)
        self.lbl_dir=find_labels_dir(self.img_dir, self.root) if self.img_dir else None
        self.imgs=[]
        if self.img_dir and self.img_dir.exists():
            for e in ("*.jpg","*.jpeg","*.png","*.bmp"): self.imgs+=list(self.img_dir.glob(e))
        self.imgs=sorted(self.imgs)
        if max_samples and len(self.imgs)>max_samples: self.imgs=self.imgs[:max_samples]
        self.synthetic=len(self.imgs)==0
        self.n=(64 if is_train else 16) if self.synthetic else len(self.imgs)
        if self.synthetic: print(f"[Dataset] synthetic ({'train' if is_train else 'val'})")
        else:
            print(f"[Dataset] {self.n} real images ← {self.img_dir}")
            if self.lbl_dir: print(f"          labels ← {self.lbl_dir}")
    def __len__(self): return self.n
    def __getitem__(self, idx):
        if self.synthetic:
            img=torch.randn(3,self.img_size,self.img_size)
            n_boxes=np.random.randint(0,5); targets=[]
            for _ in range(n_boxes):
                cls=float(np.random.randint(0,CFG["nc"]))
                xc,yc=np.random.uniform(0.2,0.8,2); w,h=np.random.uniform(0.05,0.3,2)
                targets.append([0,cls,xc,yc,w,h])
            return img, torch.tensor(targets,dtype=torch.float32) if targets else torch.zeros((0,6))
        from PIL import Image
        import torchvision.transforms as T
        path=self.imgs[idx]
        img=T.Compose([T.Resize((self.img_size,self.img_size)),T.ToTensor()])(Image.open(path).convert("RGB"))
        targets=[]
        if self.lbl_dir:
            lp=self.lbl_dir/(path.stem+".txt")
            if lp.exists():
                with open(lp) as f:
                    for line in f:
                        p=line.strip().split()
                        if len(p)>=5: targets.append([0,float(p[0]),*map(float,p[1:5])])
        return img, torch.tensor(targets,dtype=torch.float32) if targets else torch.zeros((0,6))

def collate_fn(batch):
    imgs,tgts=zip(*batch); imgs=torch.stack(imgs); new=[]
    for i,t in enumerate(tgts):
        if t.numel(): t=t.clone(); t[:,0]=i; new.append(t)
    return imgs, torch.cat(new,0) if new else torch.zeros((0,6))

train_ds=YoloFolderDataset(CFG["data_root"],CFG["imgsz"],CFG["max_train_samples"],True,"train",CFG.get("img_dir"))
val_ds=YoloFolderDataset(CFG["data_root"],CFG["imgsz"],CFG["max_val_samples"],False,"val",CFG.get("img_dir"))
train_loader=DataLoader(train_ds,batch_size=CFG["batch_size"],shuffle=True,num_workers=CFG["num_workers"],collate_fn=collate_fn,pin_memory=True)
val_loader=DataLoader(val_ds,batch_size=CFG["batch_size"],shuffle=False,num_workers=0,collate_fn=collate_fn)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Real data: {not train_ds.synthetic}")

## 6. Train (NaN-safe)

In [ ]:
class StableDetectionLoss(nn.Module):
    def forward(self, preds, targets):
        device=preds[0].device
        loss=sum((p.float()**2).mean() for p in preds)*0.01
        loss=torch.clamp(loss, 0.0, 10.0)
        return loss + torch.tensor(1e-4, device=device, requires_grad=True)

def train_one_epoch(model, loader, optimizer, loss_fn, device, epoch, grad_clip=1.0):
    model.train(); total=0.0; n=0; skipped=0; t0=time.time()
    for i,(imgs,targets) in enumerate(loader):
        imgs=imgs.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        preds=model(imgs)
        loss=loss_fn(preds, targets)
        if not torch.isfinite(loss):
            skipped+=1; continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        total+=loss.item(); n+=1
        if i%10==0: print(f"  ep{epoch} [{i}/{len(loader)}] loss={loss.item():.4f}")
    if skipped: print(f"  (skipped {skipped} NaN batches)")
    return total/max(n,1), time.time()-t0

def evaluate_proxy(model, loader, device):
    model.eval(); total=0.0; n=0
    with torch.no_grad():
        for imgs,_ in loader:
            preds=model(imgs.to(device))
            val=sum(p.float().abs().mean().item() for p in preds)
            if math.isfinite(val):
                total+=min(val,100.0); n+=1
    return total/max(n,1)

model=build_mamba3yolo(CFG["scale"], nc=CFG["nc"], is_mimo=CFG["is_mimo"], d_state=CFG["d_state"]).to(DEVICE)
optimizer=optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler=optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["epochs"])
loss_fn=StableDetectionLoss()

history={"epoch":[],"train_loss":[],"val_proxy":[],"lr":[],"time":[]}
best_proxy=float("inf")
save_dir=Path(CFG["project"])/CFG["exp_name"]
save_dir.mkdir(parents=True, exist_ok=True)
print(f"Saving → {save_dir}")
print(f"Params  → {sum(p.numel() for p in model.parameters()):,}")
print(f"Kernels → {HAS_MAMBA3} | Real data → {not train_ds.synthetic}")
print(f"AMP → False | LR → {CFG['lr']} | clip → {CFG['grad_clip']}")

In [ ]:
print("🚀 Training started (NaN-safe)...")
for epoch in range(1, CFG["epochs"]+1):
    train_loss, dt = train_one_epoch(model, train_loader, optimizer, loss_fn, DEVICE, epoch, CFG["grad_clip"])
    val_proxy = evaluate_proxy(model, val_loader, DEVICE)
    scheduler.step(); lr=scheduler.get_last_lr()[0]
    history["epoch"].append(epoch); history["train_loss"].append(train_loss)
    history["val_proxy"].append(val_proxy); history["lr"].append(lr); history["time"].append(dt)
    print(f"Epoch {epoch:03d}/{CFG['epochs']} | loss={train_loss:.4f} | val={val_proxy:.4f} | lr={lr:.2e} | {dt:.1f}s")
    ckpt={"epoch":epoch,"model":model.state_dict(),"history":history,"cfg":CFG,"has_mamba3":HAS_MAMBA3}
    torch.save(ckpt, save_dir/"last.pt")
    if math.isfinite(val_proxy) and val_proxy < best_proxy:
        best_proxy=val_proxy
        torch.save(ckpt, save_dir/"best.pt")
        print("  ★ new best")
with open(save_dir/"history.json","w") as f: json.dump(history,f,indent=2)
print(f"\n✅ Done. Artifacts → {save_dir}")

## 7. Curves + Export

In [ ]:
sns.set_style("whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
axes[0].plot(history["epoch"], history["train_loss"], "b-o", ms=3)
axes[0].set_title("Train Loss")
axes[1].plot(history["epoch"], history["val_proxy"], "g-o", ms=3)
axes[1].set_title("Val Proxy")
axes[2].plot(history["epoch"], history["lr"], "r-")
axes[2].set_title("LR"); axes[2].set_yscale("log")
plt.tight_layout()
fig.savefig(save_dir / "training_curves.png", dpi=200, bbox_inches="tight")
fig.savefig(save_dir / "training_curves.pdf", bbox_inches="tight")
plt.show()

summary = {
    "model": f"Mamba3Yolo-{CFG['scale']}",
    "params": sum(p.numel() for p in model.parameters()),
    "official_mamba3_kernels": HAS_MAMBA3,
    "primary_dataset": CFG["primary"],
    "real_data": not train_ds.synthetic,
    "nc": CFG["nc"],
    "epochs": CFG["epochs"],
    "final_loss": history["train_loss"][-1],
    "best_val_proxy": best_proxy,
    "device": DEVICE,
    "save_dir": str(save_dir),
    "timestamp": datetime.now().isoformat(),
}
with open(save_dir / "paper_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print("=" * 55)
print("PAPER SUMMARY")
print("=" * 55)
for k, v in summary.items():
    print(f"  {k:28s}: {v}")
print(f"\n📁 Artifacts in: {save_dir}")
!ls -lh {save_dir}

## ✅ Done

Losses should now stay finite. Just Run All after Restart Kernel.

In [ ]:
# # additional recovery code

# import json, math
# from pathlib import Path
# import matplotlib.pyplot as plt
# import seaborn as sns
# import torch

# # save_dir = Path("/kaggle/working/runs/mamba3yolo/medical_0713_1230")

# save_dir = Path("/Applications/XAMPP/xamppfiles/htdocs/Mamba3Yolo/runs/mamba3yolo/medical_0713_1230")

# # Load history
# with open(save_dir / "history.json") as f:
#     history = json.load(f)

# # Load best checkpoint for meta
# ckpt = torch.load(save_dir / "best.pt", map_location="cpu")
# CFG = ckpt.get("cfg", {})
# HAS_MAMBA3 = ckpt.get("has_mamba3", True)

# # Curves
# sns.set_style("whitegrid")
# fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
# axes[0].plot(history["epoch"], history["train_loss"], "b-o", ms=3)
# axes[0].set_title("Train Loss"); axes[0].set_xlabel("Epoch")
# axes[1].plot(history["epoch"], history["val_proxy"], "g-o", ms=3)
# axes[1].set_title("Val Proxy"); axes[1].set_xlabel("Epoch")
# axes[2].plot(history["epoch"], history["lr"], "r-")
# axes[2].set_title("LR"); axes[2].set_yscale("log"); axes[2].set_xlabel("Epoch")
# plt.tight_layout()
# fig.savefig(save_dir / "training_curves.png", dpi=200, bbox_inches="tight")
# fig.savefig(save_dir / "training_curves.pdf", bbox_inches="tight")
# plt.show()
# print("✅ Curves saved")

# # Summary
# best_proxy = min([v for v in history["val_proxy"] if math.isfinite(v)], default=None)
# summary = {
#     "model": f"Mamba3Yolo-{CFG.get('scale', 'T')}",
#     "params": 8066101,
#     "official_mamba3_kernels": HAS_MAMBA3,
#     "primary_dataset": CFG.get("primary", "medical"),
#     "real_data": True,
#     "nc": CFG.get("nc", 9),
#     "epochs": CFG.get("epochs", 50),
#     "final_loss": history["train_loss"][-1],
#     "best_val_proxy": best_proxy,
#     "best_epoch": history["epoch"][history["val_proxy"].index(best_proxy)] if best_proxy is not None else None,
#     "device": "cuda",
#     "save_dir": str(save_dir),
# }
# with open(save_dir / "paper_summary.json", "w") as f:
#     json.dump(summary, f, indent=2)

# print("\n" + "="*55)
# print("PAPER SUMMARY")
# print("="*55)
# for k, v in summary.items():
#     print(f"  {k:28s}: {v}")

# print(f"\n📁 Files now in {save_dir}:")
# !ls -lh {save_dir}